In [ ]:
#Installing the transformers and dataset
! pip install datasets transformers
! pip install -U datasets huggingface_hub fsspec

  Using cached fsspec-2025.5.1-py3-none-any.whl.metadata (11 kB)


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset

# Load the Yelp_reviw_dataset which is for sentiment analysis and fine tune llm on review style
datasets = load_dataset("yelp_review_full", split="train[:30000]")



In [ ]:
from transformers import pipeline, GPT2Tokenizer, GPT2LMHeadModel

# Using the GPT2Tokenizer for fine tuning
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
pretrained_model = GPT2LMHeadModel.from_pretrained("gpt2")

generator_before = pipeline("text-generation", model = pretrained_model, tokenizer = tokenizer)
tokenizer.pad_token = tokenizer.eos_token

# Tokenizing the dataset
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = datasets.map(tokenize_function, batched=True, num_proc = 4, remove_columns=["text"])

Device set to use cuda:0


Map (num_proc=4):   0%|          | 0/17000 [00:00<?, ? examples/s]

In [ ]:
tokenized_datasets = tokenized_datasets.filter(lambda example: len(example["input_ids"]) > 0) # Removed redundant filtering

block_size = 128  # Getting 128 tokens at a time in each training example



def group_texts(examples):
    # Flatten all fields (e.g., input_ids and attention_mask)
    concatenated_examples = {}
    for k in examples.keys():

        if isinstance(examples[k][0], list):
            concatenated_examples[k] = sum(examples[k], [])
        else:
            concatenated_examples[k] = examples[k]


    total_length = len(concatenated_examples["input_ids"])

    # Drop remainder to get only full blocks
    total_length = (total_length // block_size) * block_size

    if total_length == 0:
        return {}  # Drop if nothing is long enough

    result = {}
    for k in concatenated_examples.keys():
        result[k] = [
            concatenated_examples[k][i : i + block_size]
            for i in range(0, total_length, block_size)
        ]

    # Add labels explicitly
    result["label"] = result["input_ids"].copy()

    return result

Filter:   0%|          | 0/17000 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForLanguageModeling
# data collator takes a list of examples from dataset and turns them into a properly formatted, padded, batched input for model.
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


# Appling the grouping to all the text
final_dataset = tokenized_datasets.map(group_texts, batched=True, batch_size =1000, num_proc=4,remove_columns=tokenized_datasets.column_names )
print(len(final_dataset))
for i in range(len(final_dataset)):
    if len(final_dataset[i]['input_ids']) != 128:
        print(i)
split_dataset = final_dataset.train_test_split(test_size=0.1)

Map (num_proc=4):   0%|          | 0/17000 [00:00<?, ? examples/s]

17000


In [ ]:
from transformers import GPT2LMHeadModel, Trainer, TrainingArguments
model = GPT2LMHeadModel.from_pretrained("gpt2")

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=1,
    eval_strategy = "epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = split_dataset["train"],
    eval_dataset=split_dataset["test"],
    tokenizer = tokenizer,
    data_collator = data_collator

)

trainer.train()

# import math
# eval_results = trainer.evaluate()
# print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

/tmp/ipython-input-8-4281869697.py:13: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: jangid-05-yogesh (jangid-05-yogesh-iitb) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.499800,3.415200


TrainOutput(global_step=1913, training_loss=3.5258890521881536, metrics={'train_runtime': 682.1114, 'train_samples_per_second': 22.43, 'train_steps_per_second': 2.805, 'total_flos': 999442022400000.0, 'train_loss': 3.5258890521881536, 'epoch': 1.0})

In [ ]:
prompt = "The resturant was"

# Getting the output before fine tuning the GPT-2
output_before = generator_before(prompt, max_length=80, num_return_sequences=1)[0]['generated_text']
print("\n\n\n")
print("Before fine-tuning:\n", output_before)

# Saving the fine tuned model
trainer.save_model("gpt2_finetuned")
tokenizer.save_pretrained("gpt2_finetuned")

from transformers import GPT2LMHeadModel, GPT2Tokenizer

fine_tuned_model = GPT2LMHeadModel.from_pretrained("gpt2_finetuned")
fine_tuned_tokenizer = GPT2Tokenizer.from_pretrained("gpt2_finetuned")
# Getting the output after fine tuning
generator_after = pipeline("text-generation", model=fine_tuned_model, tokenizer=fine_tuned_tokenizer)
output_after = generator_after(prompt, max_length=80, num_return_sequences=1)[0]['generated_text']
print("\n\n\n")
print("After fine-tuning:\n", output_after)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)






Before fine-tuning:
 The resturant was the most obvious; the one located in the center of the building, its floor was covered with a thick layer of rubble. The building itself was not so obvious, and it was not even obvious that the building was covered with the same layer of rubble as the resturant. The second was the second room, the first was the second room, and the third room was the third room.

The first was the second room with the floor covered with the debris of the first room and the second room with the floor covered with the debris of the second room.

The second room was the second room with the floors covered with the debris of the first room and the third room.

The third room was the third room with the floor covered with the debris of the first room and the second room with the floor covered with the debris of the second room.

The third room was the third room with the floors covered with the debris of the first room and the second room with the floor covered wit

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)






After fine-tuning:
 The resturant was pretty good, and I like the fact that they have more employees than the other two restaurants in the park. They have a lot of staff and the staff is always friendly, and the food is good. Definitely a must try for any restaurant owner.\n\nOne thing that annoys me is the fact that I have to go on a Saturday, but the food is pretty good. The chicken was pretty good, and the salad was pretty good. The fish was pretty good. The chicken was really good, but the bread was pretty bad. Overall, I'm a huge fan of the restaurant, and I am very open to changing menu items. I love the fact that the staff is always friendly, and the food is good. I definitely will come back for more of the traditional dining experiences.\n\nI was very impressed by the location and the food. The fish was pretty good, but the salad was pretty good. The bread was really pretty good, but the bread was really bad. The wine list was really short, but it was pretty good. The place

In [ ]:
# Showing the perplexity of trained model

import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 30.42
